<a href="https://colab.research.google.com/github/lorenasandoval88/polygenic_risk_scores/blob/main/colab_notebooks/1000gen_ref_1K.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

What this code is doing (big picture)

It creates a small, random set of ~1,000 SNPs from the 1000 Genomes Project

- across all chromosomes
- only common variants
- includes genotypes for all individuals

In [ ]:
#1. Install tools in Colab
!apt-get update -qq
!apt-get install -y bcftools tabix wget

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
wget is already the newest version (1.21.2-2ubuntu1.1).
The following additional packages will be installed:
  libhts3 libhtscodecs2
Suggested packages:
  python3-numpy python3-matplotlib texlive-latex-recommended
The following NEW packages will be installed:
  bcftools libhts3 libhtscodecs2 tabix
0 upgraded, 4 newly installed, 0 to remove and 58 not upgraded.
Need to get 1,491 kB of archives.
After this operation, 4,626 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhtscodecs2 amd64 1.1.1-3 [53.2 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhts3 amd64 1.13+ds-2build1 [390 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/unive

In [ ]:
#2. Before running the loop, you must download all chromosomes, not just chr22.
# do once
%%bash
for CHR in {1..22}
do
  wget -q https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr${CHR}.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz
  wget -q https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/release/20130502/ALL.chr${CHR}.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz.tbi
done

In [ ]:
%%bash
# ~50 SNPs × 20 chromosomes = 1000 SNPs
# Step 1 — loop across chromosomes
rm -f combined_1000snps.tsv

for CHR in {1..22}
do
  echo "Processing chr${CHR}..."

  bcftools view -m2 -M2 -v snps \
    ALL.chr${CHR}.phase3_shapeit2_mvncall_integrated_v5b.20130502.genotypes.vcf.gz \
  | bcftools view -i 'MAF>0.05' \
  | bcftools query -f '%CHROM\t%POS\t%ID\t%REF\t%ALT[\t%GT]\n' \
  | shuf -n 50 >> combined_1000snps.tsv
done